# Forecast opponent skills with target-relative missiles and HGT edge attention

Run **24_prepare_target_relative_missile_nn_duel_skill_forecasting.ipynb** first.
This notebook ingests its verified **v2 115-channel dataset**, including target-relative
missile positions and velocities. Existing v2 prepared data can be reused. Earlier
position-only datasets need fresh preparation. Start fresh HGT training runs;
previous typed-message-passing or HGT-v1 checkpoints cannot be resumed as the
new HGT-v2 model. Older checkpoints still reload through their saved configuration.

Each frame contains two aircraft nodes and up to eight optional missile nodes.
Four node types distinguish observer aircraft, opponent aircraft, friendly missile
and hostile missile. Observer and opponent inputs use separate encoders; missiles
share their input encoder and receive distinct friendly/hostile role embeddings.
The observer node includes its current selected skill and causal acceleration;
the opponent node includes its relative state, energy and causal acceleration.
Friendly missile position and velocity are relative to the opponent; hostile
missile position and velocity are relative to the observer. Edge geometry restores
common positions and world velocities, without supplying absolute observer position.

The spatial encoder uses **two Heterogeneous Graph Transformer (HGT) layers**, each
with **width 64 and four attention heads**. Each node type has separate query, key,
value and output projections. Each of five directed relation types has its own
key/attention and value/message transform, with learned per-head priorities for
(source type, relation, destination type). Typed output projections, learned
residual gates and LayerNorm update node states. **HGT model version 2** enables
separate aggregation of three relation families: aircraft, ownership and targeting.
Attention softmax is computed within each incoming family at each destination/head.
Each family gets its own message summary. A learned gate reads the current node,
the family summary and a family embedding, then normalizes gate weights over only
present families. This keeps missile count from crowding aircraft messages out of
the same softmax. Empty families and absent missiles receive exactly zero weight.

The aircraft pair and every missile–aircraft pair have edges in both directions.
Relations distinguish aircraft–aircraft, ownership in either direction, and
missile–opposing-aircraft in either direction. This gives up to **34 directed edges**.
No missile–missile edges or persistent slot identities are assumed. In this 1-v-1
corpus the opposing aircraft is each missile's target; ownership does not establish
an observed seeker lock. Every directed edge now carries **18 numerical features**:

| Features | Scaling / meaning |
| --- | --- |
| Relative position (3), velocity (3), distance, closing speed | Existing physical scales: [50000, 50000, 10000] m, 1200 m/s, 50000 m, 1200 m/s |
| Time and distance at closest approach | Current relative-motion extrapolation, bounded to 0–30 s; divided by 30 s and 50000 m |
| Bearing alignment | Cosine between source world velocity and the line of sight toward the destination; zero if direction is undefined |
| Relative speed | Magnitude of relative velocity / 1200 m/s |
| Source kinetic, potential and total energy (3) | Specific energies in J/kg, divided by 1e6 |
| Destination kinetic, potential and total energy (3) | Same units and scale |

Closest approach uses `t_cpa = clip(-r·v / max(v·v, 1), 0, 30)` and
`d_cpa = ||r + t_cpa v||`, with metres and seconds. It is a constant-velocity
estimate from the **current frame**, not a simulated future trajectory.
Specific kinetic energy is `0.5 * ||world_velocity||**2`; potential energy is
`9.80665 * altitude`; total energy is their sum. Masses are not recorded, so energy
is per unit mass. Aircraft altitude and velocity already exist in the v2 inputs;
missile altitude and velocity follow from their relative tracks. No dataset change
or new privileged input is required. Learned edge projections use all 18 features
for both attention logits and message values.

**Learnable threat prior:** evaluate each missile's canonical missile→target edge.
The initial physical log-score is

`p * log((d_min + f) / (d_i + f)) - log(1 + d_cpa/f) - log(1 + t_cpa/tau) + alignment`,

where `f=1000 m`, `p=1`, `tau=5 s`, and `d_min` considers only present missiles to
the same target. A trainable **nonnegative strength per graph head** multiplies
this physical score. An MLP taking all 18 canonical edge features adds a learned
per-head residual, initialized to zero. Thus closest approach and bearing influence
the initial prior, while energy, speed and the other features can adjust it through
learning. Scores are shifted to give the strongest present threat log-prior zero
within each target group. The resulting bias applies to both directions of each
missile–target edge; ownership and aircraft edges retain zero log-prior. It is added
to HGT/geometry logits before **within-family** attention. Family gates then combine
the three summaries. Receding missiles can rank below farther missiles on an
intercept course. Empty slots are masked throughout; no-missile frames stay finite.

After spatial encoding, both aircraft embeddings and separate masked friendly/
hostile mean/max/count summaries give one vector per frame. The temporal model
retains **width 96, six heads, two layers, dropout 0.25** and **51 past frames**.
Graph attention is per frame; temporal attention operates only within past history.
Both acceleration estimates use backward five-frame/0.5-second differences.
No future observations, opponent decisions, pilot identity or outcomes enter inputs.

Training retains seeds **42/43/44**, up to 100 epochs, capped inverse-square-root
class weights and L1 `1e-6`. **AdamW warm-up and cosine decay** replace the
constant learning rate: the first 5% of planned optimizer updates ramp from 10% of
the peak `3e-4` to the peak, then cosine decay reaches 5% of peak (`1.5e-5`) on the
last planned update. One-update smoke runs use the peak. The schedule advances
only after successful optimizer updates and persists its state and update count
in checkpoints. Both ordinary early stopping and sustained-overtraining checks
are deferred until after epoch 20, allowing warm-up and initial decay. Patience
remains 10 stale epochs / 3 overtraining epochs. Best checkpoints are still selected
from epoch one; a stopped run may finish before the schedule reaches its floor. The grouped
partitions, disjoint 56-raw-frame training histories and exact +1/+2/+5-second
opponent-skill labels are unchanged. Unweighted test NLL selects checkpoints;
validation is evaluated only after every seed finishes selection.

**Run All trains three independent HGT models and saves diagnostics for each.**
Install `.[ml]` and a Jupyter/IPython kernel; no graph-library dependency is needed.
CUDA can use BF16 projections; graph attention/softmax/reductions use FP32.
Evaluation uses FP32/math attention. CPU uses FP32. Every code cell is timed.
For saved reports, set `TRAIN_MODEL=False`, supply every `EXISTING_RUNS` path and
leave `REEVALUATE_SAVED=False`.

Data, run receipts and default MLflow tracking use
`artifacts/nn_duel_target_relative_missile_forecasting/`.
Set `BVR_NN_TARGET_RELATIVE_MISSILE_DATA` to override the prepared directory, or
`BVR_NN_TARGET_RELATIVE_MISSILE_MLFLOW=0` to disable optional tracking.
HGT runs have architecture `hgt` and publish `comparisons/latest_hgt_runs.json`.
Checkpoints save the model version, graph-head count, CPA/prior settings,
learned gates and prior parameters, optimizer/scheduler state and update count,
normalization and target-relative feature flags for exact factory reconstruction.
No improvement is claimed before training and evaluation.


In [ ]:
%%time
from pathlib import Path
import os
import sys

root_override = os.environ.get("BVR_NN_ROOT")
candidates = ([Path(root_override).expanduser()] if root_override else
              [Path.cwd(), *Path.cwd().parents])
ROOT = next((p.resolve() for p in candidates
             if (p / "bvr_behavior_prediction/nn_duel_skill_forecasting.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Set BVR_NN_ROOT to the full Trajectory_Classification checkout path.")
sys.path.insert(0, str(ROOT))
from bvr_behavior_prediction.notebook_runtime import require_pyarrow
require_pyarrow()  # Report the kernel environment and native-library repair steps.
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from bvr_behavior_prediction import nn_duel_skill_forecasting as duel_data
from bvr_behavior_prediction.paired_skill_forecasting import (
    HORIZONS, HISTORY_STEPS, RAW_HISTORY_STEPS, SPLITS, read_json, write_json, sha256,
)
DATA_ROOT = ROOT / "artifacts/nn_duel_target_relative_missile_forecasting"
EXPERIMENT_ROOT = ROOT / "artifacts/nn_duel_target_relative_missile_forecasting"
EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)

import torch
from bvr_behavior_prediction.paired_skill_forecasting import (
    DEFAULT_TRAINING, train_model, final_evaluation, metric_tables, resolve_precision,
    fit_class_weighting, class_weight_table,
)
from bvr_behavior_prediction.paired_skill_tracking import log_run_artifacts
from bvr_behavior_prediction.enemy_missile_diagnostics import save_diagnostics

DATA_OVERRIDE = os.environ.get("BVR_NN_TARGET_RELATIVE_MISSILE_DATA")
if not DATA_OVERRIDE and not (DATA_ROOT / "latest_dataset.json").exists():
    raise FileNotFoundError("Run notebook 24 first, or set BVR_NN_TARGET_RELATIVE_MISSILE_DATA to its dataset directory.")
receipt = None if DATA_OVERRIDE else read_json(DATA_ROOT / "latest_dataset.json")
DATA = Path(DATA_OVERRIDE or receipt["directory"]).expanduser().resolve()
if receipt and sha256(DATA / "manifest.json") != receipt["manifest_sha256"]:
    raise ValueError("The prepared dataset changed since notebook 24 completed.")
data = duel_data.TargetRelativeMissileNNDuelData(DATA)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TRAINING_SEEDS = [42, 43, 44]  # One independently fitted three-horizon NN per seed.
ARCHITECTURE = "hgt"
TRAIN_MODEL = True
REEVALUATE_SAVED = False  # Explicitly enable to replace already-saved final evaluations.
RESUME = False
EXISTING_RUNS = {}  # For resume/report-only: {42: r"C:\...\runs\nn-duel-target-relative-missile-s42_...", ...}.
if not TRAINING_SEEDS or len(set(TRAINING_SEEDS)) != len(TRAINING_SEEDS):
    raise ValueError("Choose at least one distinct training seed.")
if (RESUME or not TRAIN_MODEL) and set(EXISTING_RUNS) != set(TRAINING_SEEDS):
    raise ValueError("Set EXISTING_RUNS for every selected seed when resuming or reporting.")
RUN_STAMP = pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%S%fZ")
RUNS = {seed: Path(EXISTING_RUNS[seed]).expanduser().resolve() if seed in EXISTING_RUNS
        else EXPERIMENT_ROOT / "runs" / f"nn-duel-target-relative-missile-{ARCHITECTURE}-s{seed}_{RUN_STAMP}"
        for seed in TRAINING_SEEDS}
CONFIG = dict(DEFAULT_TRAINING)
CONFIG.update(architecture=ARCHITECTURE, observer_acceleration=True, target_relative_missiles=True,
              target_relative_missile_velocities=True, width=96, heads=6, layers=2,
              graph_width=64, graph_layers=2, graph_heads=4, hgt_version=2,
              threat_cpa_horizon_s=30.0, threat_time_scale_s=5.0, threat_prior_strength=1.0,
              missile_distance_floor_m=1000.0, missile_distance_power=1.0,
              dropout=0.25, epochs=100, learning_rate=3e-4,
              lr_schedule="warmup_cosine", warmup_fraction=0.05,
              warmup_start_ratio=0.1, min_lr_ratio=0.05,
              early_stopping=True, early_stopping_min_epochs=20,
              early_stopping_patience=10, l1_lambda=1e-6,
              class_weighting="sqrt_inverse", max_class_weight_ratio=100.0,
              overtraining_shutdown=True, overtraining_min_epochs=20, overtraining_patience=3,
              overtraining_relative_increase=0.05, overtraining_min_gap=0.1,
              precision="bf16", evaluation_precision="fp32", evaluation_attention="math",
              batch_size=512,
              evaluation_batch_size=1024, data_backend="prefetch" if DEVICE == "cuda" else "direct",
              num_workers=2, cpu_threads=4, compile_model=False)
MLFLOW_ENABLED = os.environ.get("BVR_NN_TARGET_RELATIVE_MISSILE_MLFLOW", "1") == "1"
MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI",
    f"sqlite:///{(EXPERIMENT_ROOT / 'mlflow.db').as_posix()}")
TRACKING = {"enabled": MLFLOW_ENABLED, "tracking_uri": MLFLOW_TRACKING_URI,
            "experiment": "nn-duel-target-relative-missile-skill-forecasting", "log_per_class": True,
            "log_model": True,
            "artifact_root": (EXPERIMENT_ROOT / "mlflow_artifacts").as_uri()
                             if MLFLOW_TRACKING_URI.startswith("sqlite:") else None}
print("Prepared dataset:", DATA)
print("Model run directories:", {s: str(p) for s, p in RUNS.items()})
display(pd.Series({**CONFIG, "device": DEVICE,
    "effective_precision": resolve_precision(DEVICE, CONFIG["precision"]),
    "input_features": len(data.manifest["features"]), "classes": len(data.classes),
    "history_steps": HISTORY_STEPS, "horizons_s": HORIZONS}))
display(pd.DataFrame(data.manifest["counts"]).T)
if MLFLOW_ENABLED and MLFLOW_TRACKING_URI.startswith("sqlite:"):
    print(f'mlflow ui --backend-store-uri "{MLFLOW_TRACKING_URI}" --host 127.0.0.1 --port 5000')
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.cuda.synchronize()

# This audit reads only labels retained in the training partition.
CLASS_WEIGHTING = fit_class_weighting(data, CONFIG)
CLASS_WEIGHT_AUDIT = class_weight_table(CLASS_WEIGHTING)
with pd.option_context("display.max_rows", None):
    display(CLASS_WEIGHT_AUDIT.pivot(index="skill", columns="horizon_s",
                                    values=["training_support", "weight"]).round(4))
print("Zero-training-support skill/horizon pairs:")
display(CLASS_WEIGHT_AUDIT.loc[CLASS_WEIGHT_AUDIT.training_support.eq(0)])


## Fit independently seeded networks

Let `n[h,c]` be the number of retained training labels for class `c` at horizon
`h`, ignoring missing targets (`-1`), and `n_max[h]` the largest such class count.
For classes seen during training, form

`r[h,c] = min(5, sqrt(n_max[h] / n[h,c]))`

and normalise by the training-frequency weighted mean:

`w[h,c] = r[h,c] / sum_c(training_frequency[h,c] * r[h,c])`.

The rare/common per-example loss ratio is at most 5. For example, an uncapped 16:1
support ratio gives rare examples four times the weight of common examples. The
frequency-weighted mean weight is one for each horizon, keeping the expected loss
scale comparable with unweighted CE and L1. Zero-support classes receive zero
positive-target weight and retain their output logits. Their absence is audited.

For each horizon, average `w[h,y] * (-log p[h,y])` over its valid labels; then average
the represented horizons equally. The denominator is the number of valid labels,
not the minibatch sum of weights: weights remain effective even when a minibatch
contains just one class. Missing labels contribute neither loss nor gradient.
The masked weighted cross-entropy is implemented in FP32 under BF16 training too.

The loss uses independent missing-label masks. Scaling and the split are fixed by notebook 24 for every seed.
The training objective adds `l1_lambda * sum(abs(model_weights))` (default `1e-6`) to
masked class-weighted cross-entropy. This includes trainable matrix/tensor weights and position
embeddings, excluding biases and LayerNorm scales. AdamW weight decay remains 0.01.
Train/test NLL is evaluated without L1, in evaluation mode; checkpoint selection
therefore remains comparable across epochs. History records the L1 contribution,
evaluation-mode weighted training cross-entropy (`train_data_loss`), its sum with
L1 (`train_objective`), and the unweighted test-minus-train NLL gap separately.
Unweighted mean test NLL still selects checkpoints and drives both stopping rules;
the objective is not changed to favour a checkpoint based on weighted test scores.

The warm-up/cosine schedule is based on the planned number of optimizer updates,
not observed validation metrics. Its state, current optimizer learning rate and
completed-update count are saved together. Exact resume requires unchanged data,
source, epoch budget, batch size and schedule settings. Epoch history records the
first/last used learning rate, next-update rate and cumulative optimizer steps.

Stopping checks begin **after epoch 20** for this recipe. Ten stale test-NLL epochs
trigger ordinary early stopping. The independent overtraining rule requires three
consecutive epochs where test NLL exceeds its best by more than 5%, test-minus-train
NLL exceeds 0.1, and train NLL is below its value at the best-test checkpoint.
A recovery resets its counter. Checkpoint selection runs from epoch one, and both
rules preserve `best.pt`. Set both stopping switches to False to complete the full
cosine schedule. These are starting settings, not tuned performance guarantees.
Checkpoint input dimensions, feature ordering and dataset digest are stored with weights.

**Evaluation policy:** training can use CUDA BF16, while checkpoint selection and
final evaluation explicitly use `evaluation_precision="fp32"` and
`evaluation_attention="math"`. This keeps FP32 cross-entropy and unweighted NLL,
and isolates evaluation from CUDA BF16 and fused inference-attention kernels.
This is a conservative evaluation policy, not a confirmed diagnosis of the reported
GPU failure. It is held fixed across notebooks 18–20. Persistent failures write
`nonfinite_evaluation.json` in the run directory with the processing stage and
sample details; do not replace nonfinite values or discard those examples.

Start fresh timestamped runs for this schema. Run notebook 24 to prepare the new
position references; notebook-22 data and checkpoints are incompatible. Resume
requires the same source, prepared dataset, code and configuration. Reuse completed
final outputs only with `TRAIN_MODEL=False` and `REEVALUATE_SAVED=False`.


In [ ]:
%%time
histories = {}
def epoch_output(row, summary, per_class):
    print(f"Epoch {row['epoch']}: train NLL={row['train_nll']:.5f}, "
          f"test NLL={row['test_nll']:.5f}, "
          f"weighted train CE={row['train_data_loss']:.5f}, "
          f"L1={row['train_l1_penalty']:.5f}, "
          f"gap={row['generalization_gap']:.5f}, "
          f"LR={row['learning_rate']:.3g}, "
          f"overtraining streak={row['overtraining_epochs']}, {row['seconds']:.1f}s", flush=True)
    display(summary[["split", "horizon_s", "nll", "accuracy", "macro_f1",
                     "supported_macro_f1", "weighted_f1"]].round(4))
    with pd.option_context("display.max_rows", None):
        display(per_class.pivot(index=["horizon_s", "skill"], columns="split",
                                 values=["class_nll", "f1", "support"]).round(4))

for seed, run in RUNS.items():
    print(f"Training seed {seed}: {run}", flush=True)
    config = {**CONFIG, "seed": seed}
    tracking = {**TRACKING, "run_name": run.name}
    if TRAIN_MODEL:
        histories[seed] = train_model(data, run, config, callback=epoch_output,
                                     device=DEVICE, resume=RESUME, tracking=tracking)
    else:
        histories[seed] = pd.read_csv(run / "history.csv")
    status = read_json(run / "training_status.json")
    if status["status"] not in {"completed", "early_stopped"}:
        raise ValueError(f"Training has not concluded for seed {seed}.")
    saved_run = read_json(run / "run.json")
    if saved_run["data_manifest_sha256"] != sha256(DATA / "manifest.json"):
        raise ValueError(f"Saved seed {seed} belongs to a different prepared dataset.")
    if saved_run["classes"] != data.classes or saved_run["feature_names"] != data.manifest["features"]:
        raise ValueError(f"Saved seed {seed} has a different catalogue or feature ordering.")
    saved_training = dict(saved_run["training"])
    expected_training = dict(config)
    if not TRAIN_MODEL and not RESUME and not REEVALUATE_SAVED:
        # Reading completed historical outputs does not change their evaluation
        # policy. New evaluation settings may differ only in this report-only path.
        evaluation_keys = ("evaluation_precision", "evaluation_attention")
        historical_evaluation = any(saved_training.get(key) != expected_training.get(key)
                                    for key in evaluation_keys)
        required_saved_finals = ["final_metrics.csv", "final_per_class.csv", "selection.json",
                                 *[f"final_{split}.npz" for split in SPLITS]]
        if historical_evaluation and not all((run / name).is_file() for name in required_saved_finals):
            raise ValueError("Historical evaluation settings require complete saved final outputs "
                             "for report-only use; start a fresh run for the new policy.")
        for key in evaluation_keys:
            saved_training.pop(key, None)
            expected_training.pop(key, None)
    if saved_training != expected_training:
        raise ValueError(f"Requested configuration differs from saved seed {seed}.")
    display(histories[seed].tail())
if DEVICE == "cuda":
    torch.cuda.synchronize()


## Final evaluation, baseline and saved diagnostics

Every seed completes checkpoint selection before this cell reads validation.
Completed final predictions are reused unless `REEVALUATE_SAVED=True`. The
training-frequency baseline and diagnostic outputs are persisted for every seed,
then a separate receipt records the exact graph run list. No seed is chosen by
validation performance. A failed diagnostic export leaves the previous completed
receipt intact; fix the issue and rerun this cell using the same run paths.


In [ ]:
%%time
# All selected seeds finish before accessing held-out validation. Reuse completed
# final evaluations by default; report-only execution does not re-evaluate models.
all_metrics = []
required_finals = ["final_metrics.csv", "final_per_class.csv", "selection.json",
                   *[f"final_{split}.npz" for split in SPLITS]]
for seed, run in RUNS.items():
    if not REEVALUATE_SAVED and all((run / name).is_file() for name in required_finals):
        best_epoch = read_json(run / "selection.json")["epoch"]
        summary = pd.read_csv(run / "final_metrics.csv")
        print(f"Reusing saved final evaluation: {run}")
    else:
        best_epoch, summary, scores, results = final_evaluation(
            data, run, DEVICE, tracking={**TRACKING, "run_name": run.name})
        del results  # Release full predictions before the next independent seed.
    all_metrics.append(summary.assign(seed=seed, selected_epoch=best_epoch))
    display(Markdown(f"**Seed {seed}, test-selected epoch {best_epoch}**"))
    display(summary.round(4))
comparison = pd.concat(all_metrics, ignore_index=True)
for run in RUNS.values():
    comparison.to_csv(run / "seed_comparison.csv", index=False)

frequency = np.zeros((3, len(data.classes)), np.float32)
for h in range(len(HORIZONS)):
    labels = data.targets[data.indices["train"], h]
    counts = np.bincount(labels[labels >= 0], minlength=len(data.classes))
    frequency[h] = (counts + 1) / (counts.sum() + len(data.classes))
baseline_rows = []
for split in SPLITS:
    labels = data.targets[data.indices[split]]
    probabilities = np.broadcast_to(frequency, (len(labels), *frequency.shape))
    baseline, _, _ = metric_tables(labels, probabilities, data.classes, split)
    baseline_rows.append(baseline)
baseline_metrics = pd.concat(baseline_rows, ignore_index=True)
for run in RUNS.values():
    baseline_metrics.to_csv(run / "training_frequency_baseline.csv", index=False)
display(baseline_metrics.assign(model="Training frequency").round(4))

# Persist tables, calibration, sample identities, plots and a diagnostic manifest
# for EVERY seed, independently of the report seed displayed below.
for seed, run in RUNS.items():
    saved = save_diagnostics(run, data=data)
    print(f"Persisted {ARCHITECTURE}, seed {seed}: {run / 'diagnostics'}")
    if MLFLOW_ENABLED:
        log_run_artifacts(run)

# This receipt names all requested runs; no winner is chosen from final scores.
receipt_path = EXPERIMENT_ROOT / "comparisons" / f"latest_{ARCHITECTURE}_runs.json"
receipt_path.parent.mkdir(parents=True, exist_ok=True)
write_json(receipt_path, {
    "architecture": ARCHITECTURE, "runs": [str(RUNS[s]) for s in TRAINING_SEEDS],
    "seeds": TRAINING_SEEDS, "data_manifest_sha256": sha256(DATA / "manifest.json"),
    "configuration": CONFIG,
})
print("Comparison receipt:", receipt_path)
if DEVICE == "cuda":
    torch.cuda.synchronize()


## Persisted diagnostics for all seeds

Every configured run now contains diagnostic tables, calibration statistics,
confusion/reliability plots and a manifest under `diagnostics/`, alongside the
checkpoint, epoch CSVs and `final_{train,test,validation}.npz` predictions. The
saved sample identities support later aligned comparisons and whole-scenario
resampling. No notebook output needs to be retained for later comparison.

The next cell displays the first configured seed, chosen before seeing scores.
It reads the saved files rather than rendering an unrelated inline-only report.
Set `TRAIN_MODEL=False` and provide `EXISTING_RUNS` for report-only execution;
leave `REEVALUATE_SAVED=False` to reuse completed final predictions. Do not select
a new architecture or seed using held-out validation. "Test" is the checkpoint
selection partition; "validation" remains final descriptive evaluation.


In [ ]:
%%time
from IPython.display import Image

REPORT_SEED = TRAINING_SEEDS[0]  # Chosen before looking at validation results.
RUN = RUNS[REPORT_SEED]
DIAGNOSTICS = RUN / "diagnostics"
print("Saved diagnostic manifest:", DIAGNOSTICS / "manifest.json")
for table_name in ("summary.csv", "per_class.csv", "reliability.csv"):
    path = DIAGNOSTICS / table_name
    if path.is_file():
        print(table_name)
        display(pd.read_csv(path).round(4))
for path in sorted(DIAGNOSTICS.glob("*.png")):
    display(Image(filename=str(path)))
print("Completed observer-acceleration graph runs:", {s: str(p) for s, p in RUNS.items()})


## Checkpoint inference

Use the shared model factory with the saved model configuration, mean and scale,
then load `model_state`; graph/set models also retain scaling buffers in their
state dict. Input is the same normalised `(batch, 51, 115)` causal history, output
is `(batch, 3, 33)` logits. Refer to `make_model` in
`bvr_behavior_prediction.paired_skill_forecasting` for reconstruction. Retain the
checkpoint's feature ordering, catalogue and horizons. The own-skill block refers
to the observer, never the opponent.

The four enemy slots are channels 54–81; current own-skill indicators start at 82.
Live enemy-missile tracks must match the saved observer-relative North/West/Up
position/velocity scales and raw-zero absence convention. Prepared training
histories remain disjoint across both perspectives of a physical flight. No
future track, opponent decision, pilot identity or outcome enters the encoder.

For this schema, original observation channels 03–50 occupy channels 0–47.
Opponent acceleration occupies 48–50; observer acceleration occupies 51–53.
The checkpoint's `observer_acceleration=True` model setting is essential even
though both the old and new schemas have 115 features.

The saved model configuration must include `observer_acceleration=True`,
`target_relative_missiles=True` and `target_relative_missile_velocities=True`.
Supply friendly position/velocity/range relative to the opposing aircraft and
hostile position/velocity/range relative to the observer. Preserve the stored scales, raw-zero
absence, feature order and training normalization. The graph translates friendly
positions and velocities into common coordinates internally for edge calculations.

Restore architecture `hgt` through `make_model`, using the checkpoint's saved
`graph_heads`, `missile_distance_floor_m` and `missile_distance_power`. These
settings determine attention and distance weighting and must match training.

For HGT-v2 restore `hgt_version=2`, `threat_cpa_horizon_s`, `threat_time_scale_s`
and `threat_prior_strength` from `model_config`. The state dict contains the learned
family gates, prior MLP and per-head prior strengths. Configuration without
`hgt_version` reconstructs the earlier HGT-v1 architecture. The training checkpoint
also saves `scheduler_state`, `learning_rate_schedule` and `optimizer_steps`; use
`RESUME=True` with the original full run configuration to continue the schedule.
